In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from Pipeline.Global.GallstoneDataSet import GallstoneDataSet
from Pipeline.Algorithm.ExtremeLearningMachine import ExtremeLearningMachine
from Pipeline.Methodology.EvaluationMatrix import EvaluationMatrix
from Pipeline.Global.GlobalSetting import GlobalSetting

In [ ]:
model_configs = GlobalSetting.get_model_configs()

In [ ]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()
gallstone_dataset.cross_validate_test(5)

In [ ]:
testing_results = []
for config in model_configs:
    for fold_idx, (x_tr_raw, y_tr_raw, x_te_raw, y_te_raw) in enumerate(gallstone_dataset.fold_split):

        scaler = MinMaxScaler()
        x_tr_scaled = scaler.fit_transform(x_tr_raw.values)
        x_te_scaled = scaler.transform(x_te_raw.values)

        for seed in GlobalSetting.seed_test_range:
            elm = ExtremeLearningMachine(
                features_size   = x_tr_scaled.shape[1],
                hidden_size     = config["Hidden_Nodes"],
                activation_function     = config["Activation"],
                regularization_lambda   = config["Lambda_Value"]
            )
            elm.initialize_random_weights(random_seed = seed)

            elm.fit(x_tr_scaled, y_tr_raw.values)
            y_pred = elm.predict(x_te_scaled)

            evaluation = EvaluationMatrix(y_te_raw.values, y_pred)

            metrics = evaluation.get_all_metrics()
            test_result = {
                "Model_Type"   : config.get('Model_Types', 'Unknown'),
                "Hidden_Nodes" : config['Hidden_Nodes'],
                "Lambda_Value" : config['Lambda_Value'],
                "Activation"   : config['Activation'].__name__,
                "Fold_ID"      : fold_idx,
                "Seed"         : seed
            }
            test_result.update(metrics)
            testing_results.append(test_result)

In [ ]:
df_raw_baselines = pd.DataFrame(testing_results)

cols_to_keep = [
    'Accuracy', 'Precision', 'Recall', 'NPV', 'Specificity',
    'F1-Score', 'F2-Score', 'Bal Accuracy', 'MCC', 'Seed'
]

for (model_name, fold_id), group_df in df_raw_baselines.groupby(['Model_Type', 'Fold_ID']):

    file_name = f"Baseline_{model_name}_Fold_{fold_id}_Results.csv"

    final_fold_df = group_df[cols_to_keep]
    GlobalSetting.save_dataframe_to_record(final_fold_df,file_name)

In [ ]:
df_results = pd.DataFrame(testing_results)

group_cols = ["Model_Type", "Hidden_Nodes", "Lambda_Value", "Activation"]

metric_cols = [col for col in df_results.columns if col not in group_cols + ["Seed", "Fold_ID"]]

agg_funcs = {metric: ['mean','sem' ,'std', 'max' , 'min'] for metric in metric_cols}

summary_df = df_results.groupby(group_cols).agg(agg_funcs).reset_index()

summary_df.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in summary_df.columns.values
]

summary_df

In [ ]:
df_results2 = pd.DataFrame(testing_results)

group_cols = ["Model_Type", "Hidden_Nodes", "Lambda_Value", "Activation", "Fold_ID"]

metric_cols = [col for col in df_results2.columns if col not in group_cols + ["Seed"]]

agg_funcs = {metric: ['mean', 'std', 'max', 'min'] for metric in metric_cols}

summary_df_by_fold = df_results2.groupby(group_cols).agg(agg_funcs).reset_index()

summary_df_by_fold.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in summary_df_by_fold.columns.values
]

summary_df_by_fold = summary_df_by_fold.sort_values(by=["Model_Type", "Fold_ID"]).reset_index(drop=True)

summary_df_by_fold